# The Kelly criterion and risk aversion

The previous notebook (`a_risky_gamble.ipynb`) showed that a positive-arithmetic-EV gamble can wipe out almost every player when wealth compounds. The fix is to bet only a fraction $f$ of wealth each round, chosen to maximize the **expected log growth rate**
$$g(f) \;=\; E\bigl[\log\bigl(1 + f\,(R - 1)\bigr)\bigr].$$
This is the **Kelly criterion** (Kelly 1956; see also Thorp 2011, *The Kelly Capital Growth Investment Criterion*). Below we (1) compute Kelly fractions for the doubling/quartering gamble, (2) solve a Thorp-style hot/cold blackjack variant with and without a minimum-bet constraint, and (3) show how a CRRA agent's risk aversion modifies the optimal fraction.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar, minimize

## 1. The Kelly fraction

For the doubling/quartering gamble of `a_risky_gamble.ipynb` ($R = 2$ or $0.25$, each w.p. $\tfrac{1}{2}$), what fraction $f$ of wealth should be bet each round?

In [2]:
def expected_log_growth_rate(f, p, x1, x2):
    """E[log(1 + f(R-1))] for a binary asset with payoffs x1, x2 and P(R=x1)=p."""
    return p * np.log(1 + f * (x1 - 1)) + (1 - p) * np.log(1 + f * (x2 - 1))

def find_optimal_fraction(p, x1, x2):
    result = minimize_scalar(lambda f: -expected_log_growth_rate(f, p, x1, x2),
                             bounds=(0, 1), method="bounded")
    return result.x

In [3]:
p, x1, x2 = 0.5, 2.0, 0.25

fractions = [0.02, 0.15, 0.5, 1.0]
rows = [(f, expected_log_growth_rate(f, p, x1, x2)) for f in fractions]
table = pd.DataFrame(rows, columns=["Fraction f", "E[log growth]"])
print(table.to_string(index=False, float_format="{:.4f}".format))

 Fraction f  E[log growth]
     0.0200         0.0023
     0.1500         0.0102
     0.5000        -0.0323
     1.0000        -0.3466


In [4]:
f_star = find_optimal_fraction(p, x1, x2)
print(f"Optimal fraction f*           = {f_star:.4f}")
print(f"Optimal expected log growth   = {expected_log_growth_rate(f_star, p, x1, x2):.4f}")

Optimal fraction f*           = 0.1667
Optimal expected log growth   = 0.0103


$f^* = 1/6 \approx 0.1667$: the player should bet one-sixth of wealth each round, not all of it. The arithmetic expected return ($1.125$) and the Kelly-optimal expected log-growth ($\approx 0.0103$) are pointing at very different objectives.

## 2. Hot and cold decks

A blackjack card-counter sees the deck state before each hand. When the deck is **hot** the win probability is $p_{\text{hot}} = 0.52$; when it is **cold** it is $p_{\text{cold}} = 0.48$. Each hand pays $2\times$ (win) or $0$ (lose). The two cases:

1. **Unconstrained.** Choose $f_{\text{hot}}$ and $f_{\text{cold}}$ independently. The bettor opts out of unfavorable hands.
2. **Constrained.** A house minimum-bet rule forces $f_{\text{hot}} = 3\,f_{\text{cold}}$ — the bettor must keep playing cold hands at one-third the hot-hand stake.

Following Thorp (2011), §1–2.

In [5]:
p_hot, p_cold = 0.52, 0.48
x_win, x_lose = 2.0, 0.0

In [6]:
def find_hot_cold_unconstrained(p_hot, p_cold, x_win, x_lose):
    return (find_optimal_fraction(p_hot,  x_win, x_lose),
            find_optimal_fraction(p_cold, x_win, x_lose))

def find_hot_cold_constrained(p_hot, p_cold, x_win, x_lose, ratio=3.0):
    """Maximize 0.5 g(f_hot) + 0.5 g(f_cold) subject to f_hot = ratio * f_cold."""
    def neg_expected_growth(f):
        f_hot, f_cold = f
        return -(0.5 * expected_log_growth_rate(f_hot,  p_hot,  x_win, x_lose) +
                 0.5 * expected_log_growth_rate(f_cold, p_cold, x_win, x_lose))
    result = minimize(neg_expected_growth,
                      x0=[0.1, 0.1], bounds=[(0, 1), (0, 1)],
                      constraints={"type": "eq",
                                   "fun": lambda f: f[0] - ratio * f[1]})
    return tuple(result.x)

In [7]:
f_unc = find_hot_cold_unconstrained(p_hot, p_cold, x_win, x_lose)
f_con = find_hot_cold_constrained  (p_hot, p_cold, x_win, x_lose, ratio=3.0)

def overall_growth(f_hot, f_cold):
    return 0.5 * expected_log_growth_rate(f_hot,  p_hot,  x_win, x_lose) + \
           0.5 * expected_log_growth_rate(f_cold, p_cold, x_win, x_lose)

summary = pd.DataFrame({
    "f_hot (%)":         [100 * f_unc[0], 100 * f_con[0]],
    "f_cold (%)":        [100 * f_unc[1], 100 * f_con[1]],
    "E[log growth] (%)": [100 * overall_growth(*f_unc),
                          100 * overall_growth(*f_con)],
}, index=["Unconstrained", "Constrained (f_hot = 3 f_cold)"])
print(summary.to_string(float_format="{:.2f}".format))

                                f_hot (%)  f_cold (%)  E[log growth] (%)
Unconstrained                        4.00        0.00               0.04
Constrained (f_hot = 3 f_cold)       2.41        0.80               0.02


Unconstrained, the bettor sits out cold hands ($f_{\text{cold}} = 0$) and bets $4\%$ on hot ones. The minimum-bet constraint forces participation at unfavorable odds and roughly halves the long-run growth rate.

## 3. Risk aversion lowers — or raises — the Kelly fraction

Kelly betting maximizes $E[\log W]$, which is CRRA utility with relative risk aversion $\gamma = 1$. A more risk-averse agent ($\gamma > 1$) bets less; a less risk-averse one ($0 < \gamma < 1$) bets more.

We parameterize $\gamma$ via a more interpretable quantity $K$: the multiple of wealth in the win state that makes the agent indifferent to a fair coin flip whose lose state wipes out everything. Solving the indifference condition gives
$$\gamma \;=\; 1 - \frac{1}{\log_2 K}.$$
$K = 4$ corresponds to $\gamma = 0.5$ — *less* risk averse than log utility. Take a new gamble: $R = 3$ or $R = 0.1$ each w.p. $\tfrac{1}{2}$.

In [8]:
p, x1, x2 = 0.5, 3.0, 0.1
K = 4.0
gamma = 1 - 1 / np.log2(K)
print(f"K = {K}  =>  gamma = {gamma}")

K = 4.0  =>  gamma = 0.5


In [9]:
def crra_utility(w, gamma):
    """CRRA utility u(w) with RRA coefficient gamma >= 0; gamma=1 is log."""
    if gamma == 1:
        return np.log(w)
    return (w ** (1 - gamma) - 1) / (1 - gamma)

def find_optimal_fraction_crra(p, x1, x2, gamma, w0=1.0):
    """Argmax_f  E[u(w0 * (1 + f(R-1)))]  for binary R in {x1, x2}."""
    def neg_eu(f):
        w_win  = w0 * (1 + f * (x1 - 1))
        w_lose = w0 * (1 + f * (x2 - 1))
        return -(p * crra_utility(w_win, gamma) + (1 - p) * crra_utility(w_lose, gamma))
    return minimize_scalar(neg_eu, bounds=(0, 1), method="bounded").x

In [10]:
f_kelly = find_optimal_fraction(p, x1, x2)              # gamma = 1
f_crra  = find_optimal_fraction_crra(p, x1, x2, gamma)  # gamma = 0.5

comparison = pd.DataFrame({
    "gamma":           [1.0, gamma],
    "f*":              [f_kelly, f_crra],
    "E[log growth]":   [expected_log_growth_rate(f_kelly, p, x1, x2),
                        expected_log_growth_rate(f_crra,  p, x1, x2)],
}, index=["Kelly (log utility)", f"CRRA, K={K:g}"])
print(comparison.to_string(float_format="{:.4f}".format))

                     gamma     f*  E[log growth]
Kelly (log utility) 1.0000 0.3056         0.0777
CRRA, K=4           0.5000 0.6111         0.0000


The CRRA agent with $\gamma = 0.5$ bets $f^* \approx 0.61$, twice the Kelly fraction $f^* \approx 0.31$ — and *because* she is willing to bear more downside, her **expected log growth** is roughly zero rather than positive. Less risk aversion than log utility is *not* a free lunch in compounded settings.

**Takeaway.** The Kelly criterion picks $f$ to maximize expected log growth. Information (hot/cold deck states) lets you bet more on favorable hands and zero on unfavorable ones; binding constraints (minimum-bet rules) cut growth roughly in half. Risk aversion above log utility shrinks the bet; risk aversion below log utility *grows* the bet but at the cost of long-run growth — log utility is the unique level that maximizes wealth growth in compounded play.